In [1]:
## 2025.07.21 '-었-'에 대한 논문 작성느라 만듦
from pathlib import Path
dir_path = Path(r"C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치")
v_folder = dir_path / "최신버전" / "v"
sen_folder = dir_path / "최신버전" / "sen"

from datetime import datetime
time = []#걸린 시간 체크 리스트
start_t = datetime.now() #시간 체크
time.append(datetime.now() - start_t) #시간간격 저장
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M") #시작 시간 저장
v_file_list = []
sen_file_list = []
for file in v_folder.glob('*.csv'):
    print(file)
    v_file_list.append(file)

for file in sen_folder.glob('*.csv'):
    print(file)
    sen_file_list.append(file)
v_file_list = [v_file_list[2]]
sen_file_list = [sen_file_list[2]]
print(v_file_list)
print(sen_file_list)

C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\최신버전\v\국립국어원 구어 말뭉치(버전 1.2)_일부(1,15)_v5.csv
C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\최신버전\v\국립국어원 일상대화 말뭉치2020,2021_v5.csv
C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\최신버전\v\세종_문어_형태분석_말뭉치_v5.csv
C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\최신버전\sen\국립국어원 구어 말뭉치(버전 1.2)_일부(1,15)_sen2.csv
C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\최신버전\sen\국립국어원 일상대화 말뭉치2020,2021_sen2.csv
C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\최신버전\sen\세종_문어_형태분석_말뭉치_sen2.csv
[WindowsPath('C:/Users/yu2hy/OneDrive/◎2020_copus/가공_결과물/바른형태분석_말뭉치/최신버전/v/세종_문어_형태분석_말뭉치_v5.csv')]
[WindowsPath('C:/Users/yu2hy/OneDrive/◎2020_copus/가공_결과물/바른형태분석_말뭉치/최신버전/sen/세종_문어_형태분석_말뭉치_sen2.csv')]


In [2]:
import pandas as pd

#%% 1.1 v 파일 읽기: 55초 걸림.
df_v_list = []
for file in v_file_list:
    with open(file, "r", encoding = "UTF-8") as f:
        df = pd.read_csv(f, low_memory=False)
        #'Unnamed:'를 포함하는 columns 삭제
        cols_to_drop = [col for col in df.columns if 'Unnamed:' in col]
        df = df.drop(columns=cols_to_drop)
        
        df_v_list.append(df)

df_v_list[0].rename(columns={'docu_id': 'file_id'}, inplace=True)

#%% 1.2 문장 파일 읽어오기
print('%%%%%1.2 문장 파일-sen2 읽기') 
df_sen_list = []
    
for file in sen_file_list:
    with open(file, "r", encoding = "UTF-8") as f:
        df = pd.read_csv(f, low_memory=False)
        #'Unnamed:'를 포함하는 columns 삭제
        cols_to_drop = [col for col in df.columns if 'Unnamed:' in col]
        df = df.drop(columns=cols_to_drop)
        df_sen_list.append(df)
df_sen_list[0].rename(columns={'docu_id': 'file_id'}, inplace=True)

for i, df in enumerate(df_v_list):
    df['copus'] = i
    df.loc[df['V_No'] == 3035, 'V_No'] = 3025 #갖다, 가지다를 하나로 합함.
    print(f"{i}, {len(df[df['V_label']=="VX"])}")

%%%%%1.2 문장 파일-sen2 읽기
0, 574023


In [3]:
#누락값 채움. 
#object를 category형으로 바꿈. : 45초 걸림
exclude_cols = ['form/label', 'N_form', 'V_form', 'f_V_form']  # 제외할 열
for df in df_v_list:
    for col in df.columns:
        if col in exclude_cols:
            continue  # 제외할 열은 건너뛰기

        # 숫자형이면 -1로 채움
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(-1)

        # 문자열(object)이면 'NULL'로 채우고 고유값 적을 때만 category로
        elif pd.api.types.is_object_dtype(df[col]):
            df[col] = df[col].fillna('NULL')
            unique_ratio = df[col].nunique(dropna=False) / len(df)
            if unique_ratio < 0.5:
                df[col] = df[col].astype('category')

In [4]:
import numpy as np

pat_TT = r'(?:었었|았었)'
pat_T  = r'(?:었|았)'
pat_M  = r'(?:겠|겄)'

for i, df in enumerate(df_v_list):
    # EP_form
    df['EP_form'] = df['EP_form'].str.replace(' + ', '', regex=False)

    df['EP_TT'] = df['EP_form'].str.contains(pat_TT, regex=True, na=False)
    df['EP_T']  = (~df['EP_TT']) & df['EP_form'].str.contains(pat_T, regex=True, na=False)
    df['EP_M']  = df['EP_form'].str.contains(pat_M, regex=True, na=False)

    # f_EP_form
    df['f_EP_form'] = df['f_EP_form'].str.replace(' + ', '', regex=False)

    df['f_EP_TT'] = df['f_EP_form'].str.contains(pat_TT, regex=True, na=False)
    df['f_EP_T']  = (~df['f_EP_TT']) & df['f_EP_form'].str.contains(pat_T, regex=True, na=False)
    df['f_EP_M']  = df['f_EP_form'].str.contains(pat_M, regex=True, na=False)

    # self_VX
    df['self_VX'] = np.where(df['vx_no.'] == -1, False, True)


In [5]:
import numpy as np

### 추가 정보 붙이는 함수 선언     return result_df
def process_dataframe(result_df):    
    # 피벗테이블을 일반 데이터프레임으로 변환
    if 'index' in result_df.columns:
        result_df = result_df.reset_index()
    
    # category 덧붙이기
    if ('category' in result_df.columns) & ('category_0' not in result_df.columns):    
        data = {
            "category": ["강의", "강의, 학술집담회", "낭독", "뉴스", "방송대화", "방송인터뷰", "토론",
                        "교육자료", "사회일반", "신문", "인문", "일상대화", "자연", "잡지", "체험기술",
                        "총류,일반", "허구", "협력적대화"],
            "category_0": ["공적구어", "공적구어", "공적구어", "공적구어", "공적구어", "공적구어", "공적구어",
                        "문어", "문어", "문어", "문어", "사적구어", "문어", "문어", "문어",
                        "문어", "문어", "사적구어"]
        }
        df_category = pd.DataFrame(data)
        result_df = pd.merge(result_df, df_category, on="category", how="left")

    # V_label 덧붙이기
    if ('V_label' in result_df.columns) & ('V_label_0' not in result_df.columns):
        data = {
            "V_label": ["VV", "VA", "VCN", "VCP", "XSV", "XSA", "VX"],
            "V_label_0": ["VV", "VA", "VA", "VP", "VV", "VA", "VX"]
        }
        df_v_label = pd.DataFrame(data)
        result_df = pd.merge(result_df, df_v_label, on="V_label", how="left")

    # EN 정보 덧붙이기
    if ('EN_No' in result_df.columns) & ('EN_No_0' not in result_df.columns):
        result_df['EN_No'] = pd.to_numeric(result_df['EN_No'], errors='coerce')  # 'EN_No' 열을 숫자형으로 변환, 변환할 수 없는 값은 NaN으로 처리
        result_df['EN_No_0'] = result_df['EN_No'].apply(lambda x: int(x) if not np.isnan(x) else np.nan)  # 'EN_No_0' 열을 추가 (EN_No 내림)
        result_df['EN_No_1'] = result_df['EN_No'] - result_df['EN_No_0']  # 소수점 이하만 남김

    # f_EN 정보 덧붙이기
    if ('f_EN_No' in result_df.columns) & ('f_EN_No_0' not in result_df.columns):
        result_df['f_EN_No'] = pd.to_numeric(result_df['f_EN_No'], errors='coerce')  # 'EN_No' 열을 숫자형으로 변환, 변환할 수 없는 값은 NaN으로 처리
        result_df['f_EN_No_0'] = result_df['f_EN_No'].apply(lambda x: int(x) if not np.isnan(x) else np.nan)  # 'EN_No_0' 열을 추가 (EN_No 내림)
        result_df['f_EN_No_1'] = result_df['f_EN_No'] - result_df['f_EN_No_0']  # 소수점 이하만 남김
        
    # EP 정보 덧붙이기
    if ('EP_form' in result_df.columns) & ('T' not in result_df.columns):
        result_df['T'] = result_df['EP_form'].str.contains('았|었')
        result_df['M'] = result_df['EP_form'].str.contains('겠|겄')
        result_df['H'] = result_df['EP_form'].str.contains('시')

    # f_EP 정보 덧붙이기
    if ('f_EP_form' in result_df.columns) & ('f_T' not in result_df.columns):
        result_df['f_T'] = result_df['f_EP_form'].str.contains('았|었')
        result_df['f_M'] = result_df['f_EP_form'].str.contains('겠|겄')
        result_df['f_H'] = result_df['f_EP_form'].str.contains('시')

    # V_No 정보 덧붙이기
    if ('V_No' in result_df.columns) & ('V_label' not in result_df.columns):
        # V_label 생성: V_No 값에 따라 부여
        def assign_v_label(v_no):
            if 0 <= v_no < 1000:
                return 'VX'
            elif 1000 <= v_no < 2000:
                return 'VC'
            elif 2000 <= v_no < 3000:
                return 'VA'
            elif 3000 <= v_no < 5000:
                return 'VV'
            else:
                return ''  # 혹시 범위 벗어난 경우 대비

        result_df['V_label'] = result_df['V_No'].apply(assign_v_label)
    
    return result_df

### 여러 DataFrame에서 지정한 index_columns 기준으로 피벗 통계를 계산.###    return result_df
def generate_pivot_statistics(df_list, index_columns, filter_fn=None, process_fn=process_dataframe):
    """
    여러 DataFrame에서 지정한 index_columns 기준으로 피벗 통계를 계산.

    Parameters:
        df_list (list of pd.DataFrame): 분석 대상 데이터프레임 리스트
        index_columns (list of str): 피벗 기준 열 목록
        process_fn (callable, optional): 결과 df 후처리 함수 (예: process_dataframe)
        filter_fn (callable, optional): 각 df에 필터를 적용할 함수 (예: lambda df: df[df['V_label'] == 'VX'])

    Returns:
        pd.DataFrame: 통합된 피벗 통계 결과
    """
    result_df_list = []
    print(f"*** 통계를 만듭니다. ***")
    
    for i, df in enumerate(df_list):
        print(f"{i}. 시작합니다.")

        # 필터 함수가 있다면 적용
        print(f"*** {i}. 필터를 적용합니다.***")
        if filter_fn:
            df = filter_fn(df)
        
        print(f"{i}. 피벗 생성 중...")
        cross_0 = pd.pivot_table(
            df[['ID'] + index_columns],
            index=index_columns,
            aggfunc='count',
            fill_value=0,
            dropna=True,
            observed=False
        )
        cross_0 = cross_0[cross_0['ID'] != 0][['ID']]
        result_df_list.append(cross_0)

    result_df = pd.concat(result_df_list).reset_index()

    if process_fn:
        result_df = process_fn(result_df)

    return result_df

###DataFrame에서 필터를 수행한 결과를 문장과 합해서 출력 ##return result_df_list
def generate_sentence_df(df_list, df_sen_list, filter_fn=None, window=0, process_fn=None):
    """
    DataFrame에서 필터를 수행한 결과를 문장과 합해서 출력

    Parameters:
        df_list (list of pd.DataFrame): 분석 대상 데이터프레임 리스트
        df_sen_list (list of pd.DataFrame): 합할 문장 데이터프레임 리스트
        process_fn (callable, optional): 결과 df 후처리 함수 (예: process_dataframe)
        filter_fn (callable, optional): 각 df에 필터를 적용할 함수 (예: lambda df: df[df['V_label'] == 'VX'])

    Returns:
        result_df_list: list of pd.DataFrame: 통합된 데이터프레임 리스트 결과
    """
    result_df_list = []
    print(f"*** 문장 통합 df를 만듭니다. ***")
    
    for i, (df, df_sen) in enumerate(zip(df_list, df_sen_list)):
        print(f"{i}. 시작")

        # 필터 함수가 있다면 적용
        if filter_fn:
            print(f"{i}. 필터를 적용합니다.")
            taget_df = filter_fn(df)

            if window > 0:
                df = expand_window(taget_df, df, window)
            else: 
                df = taget_df
        print(f"{i}. df와 sen_df를 merge합니다.")
        df_sen = df_sen[['sen_id', 'form']]
        result_df = pd.merge(df_sen, df, how='right', on = 'sen_id')

        if process_fn:
            result_df = process_fn(result_df)
        
        print(f"{i}. 통합 완료..., 결과: {len(result_df)}")

        result_df_list.append(result_df)
    return result_df_list

###오류 후보 행들의 sen_id와 word_id2를 기준으로 주변 window만큼 확장한 행을 추출.    #return result_df
def expand_window(taget_df, df, window=2):
    """
    오류 후보 행들의 sen_id와 word_id2를 기준으로 주변 window만큼 확장한 행을 추출.
    """
    # 오류 후보들만 뽑고
    taget_row = taget_df[['sen_id', 'word_id2']].dropna().copy()
    taget_row['word_id2'] = taget_row['word_id2'].astype(int)

    # window 범위로 확장
    expanded_rows = []
    for _, row in taget_row.iterrows():
        sid = row['sen_id']
        wid = row['word_id2']
        for offset in range(-window, window + 1):
            expanded_rows.append((sid, wid + offset))

    # 중복 제거
    expanded_df = pd.DataFrame(expanded_rows, columns=['sen_id', 'word_id2']).drop_duplicates()

    # 기준 df와 merge
    df['word_id2'] = df['word_id2'].astype('Int64')  # Null 허용 정수로
    result_df = pd.merge(df, expanded_df, on=['sen_id', 'word_id2'], how='inner')
    result_df = result_df.sort_values(by=['sen_id', 'word_id2']).reset_index(drop=True)

    return result_df

In [9]:
### v에 따른 vx부류 결합
prefix = "V_VX_EP(세종문어_1101)_500"
save_folder = '2025.07.21-었'
index_columns = ['file_id', 'self_VX', 'V_No', 'vx1_No','vx_len', 'f_EP_T', 'f_EP_M','f_EP_TT', 'f_EN_No']

def filter(df):
    df_copy = df.copy()

    # --- 1) your existing filter ---
    cond = (
        (df_copy['V_No'] != -1) &
        df_copy['f_EN_No'].astype(str).str.startswith('1101', na=False) &
        (df_copy['f_EN_label'] == "EF")
    )
    # 서브셋만 떼서 작업 (가독성/안전성 ↑)
    sub = df_copy.loc[cond, ['V_No', 'vx1_No']].copy()

    # --- 1) V_No: low-frequency bucketing per range ---
    def bucket_low_freq(sub_df, col, lo, hi, cap, min_freq):
        mask = sub_df[col].between(lo, hi)
        if mask.any():
            counts = sub_df.loc[mask, col].value_counts()
            low_vals = counts[counts <= min_freq].index
            sub_df.loc[mask & sub_df[col].isin(low_vals), col] = cap

    # VA (2001~2999) -> 2999
    bucket_low_freq(sub, 'V_No', 2001, 2999, 2999, 500)
    # VV (3001~4999) -> 4999
    bucket_low_freq(sub, 'V_No', 3001, 4999, 4999, 500)

    # --- 2) vx1_No: low-frequency bucketing per range ---
    bucket_low_freq(sub, 'vx1_No', 101, 199, 199, 1000)   # 상
    bucket_low_freq(sub, 'vx1_No', 201, 499, 499, 1000)   # 부정/수혜/피사동 등
    bucket_low_freq(sub, 'vx1_No', 500, 999, 999, 1000)   # 양태

    # 서브셋 결과만 원본에 반영
    df_copy.loc[cond, ['V_No', 'vx1_No']] = sub[['V_No', 'vx1_No']]

    return df_copy[cond]

result_df = generate_pivot_statistics(
    df_list= df_v_list,
    index_columns=index_columns,
    process_fn=process_dataframe,
    filter_fn=filter
)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
print(f"***저장합니다.:    {datetime.now()}***")
out_file = dir_path / save_folder / f'{prefix}_statistics_{timestamp}.csv'
result_df.to_csv(out_file, index=True, encoding='utf-8-sig')
print(f"*** 저장 완료: {out_file} ({len(result_df):,}행) ***")

*** 통계를 만듭니다. ***
0. 시작합니다.
*** 0. 필터를 적용합니다.***
0. 피벗 생성 중...
***저장합니다.:    2025-08-09 19:49:41.051837***
*** 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\2025.07.21-었\V_VX_EP(세종문어_1101)_500_statistics_2025-08-09_19-49.csv (213,924행) ***


In [10]:
len(result_df)

213924

In [16]:
### 문장 출력

def filter(df):
    df_copy = df.copy()

    # --- 1) your existing filter ---
    base = (
        (df_copy['V_No'] != -1) &
        df_copy['f_EN_No'].astype(str).str.startswith('1101', na=False) &
        (df_copy['f_EN_label'] == "EF")
    )
    # 서브셋만 떼서 작업 (가독성/안전성 ↑)
    sub = df_copy.loc[base, ['V_No', 'vx1_No']].copy()

    # --- 1) V_No: low-frequency bucketing per range ---
    def bucket_low_freq(sub_df, col, lo, hi, cap, min_freq):
        mask = sub_df[col].between(lo, hi)
        if mask.any():
            counts = sub_df.loc[mask, col].value_counts()
            low_vals = counts[counts <= min_freq].index
            sub_df.loc[mask & sub_df[col].isin(low_vals), col] = cap

    # VA (2001~2999) -> 2999
    bucket_low_freq(sub, 'V_No', 2001, 2999, 2999, 500)
    # VV (3001~4999) -> 4999
    bucket_low_freq(sub, 'V_No', 3001, 4999, 4999, 500)

    # --- 2) vx1_No: low-frequency bucketing per range ---
    bucket_low_freq(sub, 'vx1_No', 101, 199, 199, 1000)   # 상
    bucket_low_freq(sub, 'vx1_No', 201, 499, 499, 1000)   # 부정/수혜/피사동 등
    bucket_low_freq(sub, 'vx1_No', 500, 999, 999, 1000)   # 양태

    # 서브셋 결과만 원본에 반영
    df_copy.loc[base, ['V_No', 'vx1_No']] = sub[['V_No', 'vx1_No']]
    
    # 2) Final filter: EF+1101 subset AND exclude bucketed caps
    cond = (
        (df_copy['V_No'] != -1) &
        df_copy['f_EN_No'].astype(str).str.startswith('1101', na=False) &
        (df_copy['f_EN_label'] == "EF") &
        ~df_copy['V_No'].isin([2999, 4999]) &
        ~df_copy['vx1_No'].isin([199, 499, 999])
    )
    return df_copy[cond]

result_df_list = generate_sentence_df(
    df_list=df_v_list,
    df_sen_list=df_sen_list,
    process_fn=process_dataframe,
    filter_fn=filter,
    window=0
)
## 문장 파일 각각 저장하기.
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
print(f"***저장합니다.:    {datetime.now()}***")

for i, result_df in enumerate(result_df_list):
    out_file = dir_path / save_folder / f'{prefix}_sentence_{i}_{timestamp}.csv'
    result_df.to_csv(out_file, index=True, encoding='utf-8-sig')
    print(f"*** 저장 완료: {out_file} ({len(result_df):,}행) ***")

*** 문장 통합 df를 만듭니다. ***
0. 시작
0. 필터를 적용합니다.
0. df와 sen_df를 merge합니다.
0. 통합 완료..., 결과: 774790
***저장합니다.:    2025-08-09 20:09:50.600139***
*** 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\2025.07.21-었\V_VX_EP(세종문어_1101)_500_sentence_0_2025-08-09_20-09.csv (774,790행) ***


In [8]:
### v에 따른 vx부류 결합
prefix = "V_VX_EP_EN(세종문어_1101)_500"
save_folder = '2025.07.21-었'
index_columns = ['file_id', 'V_No', 'vx_no.','vx_len', 'EP_T', 'EP_M', 'EP_TT']

def filter(df):
    df_copy = df.copy()

    # --- 1) your existing filter ---
    cond = (
        (df_copy['V_No'] != -1) &
        (df_copy['EN_No'] != -1) &
        df_copy['f_EN_No'].astype(str).str.startswith('1101', na=False) &
        (df_copy['f_EN_label'] == "EF")
    )
    # 서브셋만 떼서 작업 (가독성/안전성 ↑)
    sub = df_copy.loc[cond, ['V_No', 'vx_no.']].copy()

    # --- 1) V_No: low-frequency bucketing per range ---
    def bucket_low_freq(sub_df, col, lo, hi, cap, min_freq):
        mask = sub_df[col].between(lo, hi)
        if mask.any():
            counts = sub_df.loc[mask, col].value_counts()
            low_vals = counts[counts <= min_freq].index
            sub_df.loc[mask & sub_df[col].isin(low_vals), col] = cap

    # VA (2001~2999) -> 2999
    bucket_low_freq(sub, 'V_No', 2001, 2999, 2999, 500)
    # VV (3001~4999) -> 4999
    bucket_low_freq(sub, 'V_No', 3001, 4999, 4999, 500)

    # --- 2) vx1_No: low-frequency bucketing per range ---
    bucket_low_freq(sub, 'vx_no.', 101, 199, 199, 1000)   # 상
    bucket_low_freq(sub, 'vx_no.', 201, 499, 499, 1000)   # 부정/수혜/피사동 등
    bucket_low_freq(sub, 'vx_no.', 500, 999, 999, 1000)   # 양태

    # 서브셋 결과만 원본에 반영
    df_copy.loc[cond, ['V_No', 'vx_no.']] = sub[['V_No', 'vx_no.']]

    return df_copy[cond]

result_df = generate_pivot_statistics(
    df_list= df_v_list,
    index_columns=index_columns,
    process_fn=process_dataframe,
    filter_fn=filter
)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
print(f"***저장합니다.:    {datetime.now()}***")
out_file = dir_path / save_folder / f'{prefix}_statistics_{timestamp}.csv'
result_df.to_csv(out_file, index=True, encoding='utf-8-sig')
print(f"*** 저장 완료: {out_file} ({len(result_df):,}행) ***")

*** 통계를 만듭니다. ***
0. 시작합니다.
*** 0. 필터를 적용합니다.***
0. 피벗 생성 중...
***저장합니다.:    2025-08-09 21:16:40.589689***
*** 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\2025.07.21-었\V_VX_EP_EN(세종문어_1101)_500_statistics_2025-08-09_21-16.csv (137,278행) ***
